# NER Scheduling Model Test
Entities: `task_name`, `date`, `time`, `priority`, `status`, `person`, `duration`, `day`, `new_taskname`, `new_date`, `new_time`, `new_priority`, `new_status`, `new_duration`

NER MODEL LOADING

In [2]:
import torch
from transformers import DistilBertTokenizerFast, DistilBertForTokenClassification

MODEL_PATH = r'D:\FYP_20March\Eva_AI_Server\models\ner1_model'

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_PATH)
model     = DistilBertForTokenClassification.from_pretrained(MODEL_PATH)
model.eval()

id2label = {int(k): v for k, v in model.config.id2label.items()}
print('Model loaded.')
print('Labels:', list(id2label.values()))

Model loaded.
Labels: ['O', 'B-task_name', 'I-task_name', 'B-date', 'I-date', 'B-time', 'I-time', 'B-priority', 'I-priority', 'B-status', 'I-status', 'B-person', 'I-person', 'B-duration', 'I-duration', 'B-day', 'I-day', 'B-new_taskname', 'I-new_taskname', 'B-new_date', 'I-new_date', 'B-new_time', 'I-new_time', 'B-new_duration', 'I-new_duration', 'B-new_priority', 'I-new_priority', 'B-new_status', 'I-new_status']


OLD MODEL LOADING

In [6]:
import torch
from transformers import BertTokenizerFast, BertForTokenClassification
MODEL_PATH = r'D:\FYP_20March\Eva_AI_Server\models\ner_model'
tokenizer = BertTokenizerFast.from_pretrained(MODEL_PATH)
model     = BertForTokenClassification.from_pretrained(MODEL_PATH)

id2label = {int(k): v for k, v in model.config.id2label.items()}
print('Model loaded.')
print('Labels:', list(id2label.values()))

Model loaded.
Labels: ['O', 'B-date', 'I-date', 'B-day', 'I-day', 'B-duration', 'I-duration', 'B-new_date', 'I-new_date', 'B-new_duration', 'I-new_duration', 'B-new_priority', 'I-new_priority', 'B-new_status', 'I-new_status', 'B-new_taskname', 'I-new_taskname', 'B-new_time', 'I-new_time', 'B-person', 'I-person', 'B-priority', 'I-priority', 'B-status', 'I-status', 'B-task_name', 'I-task_name', 'B-time', 'I-time']


In [7]:
def extract_entities(text, model, tokenizer, id2label):
    words = text.lower().split()
    enc   = tokenizer(
        words,
        is_split_into_words = True,
        return_tensors      = 'pt',
        truncation          = True,
        max_length          = 128,
    )
    with torch.no_grad():
        logits = model(**enc).logits
    pred_ids = logits.argmax(dim=-1).squeeze(0).tolist()
    word_ids = enc.word_ids(batch_index=0)

    # First subword per word only
    word_labels = {}
    for tok_idx, word_id in enumerate(word_ids):
        if word_id is None or word_id in word_labels:
            continue
        word_labels[word_id] = id2label.get(pred_ids[tok_idx], 'O')

    # Build entity spans from BIO
    entities = []
    current_entity = None
    current_words  = []
    for wi in range(len(words)):
        label = word_labels.get(wi, 'O')
        if label.startswith('B-'):
            if current_entity:
                entities.append((' '.join(current_words), current_entity))
            current_entity = label[2:]
            current_words  = [words[wi]]
        elif label.startswith('I-') and current_entity == label[2:]:
            current_words.append(words[wi])
        else:
            if current_entity:
                entities.append((' '.join(current_words), current_entity))
            current_entity = None
            current_words  = []
    if current_entity:
        entities.append((' '.join(current_words), current_entity))

    return entities

print('extract_entities() ready.')

extract_entities() ready.


In [8]:
# ── Add your test inputs here ──────────────────────────────────────
inputs = [
    # ── Priority edge cases (the known issue) ──
    "add math exam on monday high priority",
    "urgent add phy exam tomorrow",
    "low priority delete cs lab",
    "add fyp meeting friday top priority",
    "add bio exam tuesday medium",

    # ── Status edge cases ──
    "mark physics exam as done",
    "mark cs lab complete",
    "physics exam done mark it",
    "all pending tasks show me",
    "delete all completed tasks",

    # ── Person vs task_name distinction ──
    "add urdu exam on monday",
    "add Ali project on tuesday",
    "add Hassan assignment on friday",
    "add cs lab with Ali on tuesday at 3pm",
    "add lab report collaborator is Hassan on wednesday",
    "add omar khayyam essay on thursday",
    "add islamiyat paper tomorrow",

    # ── Casual / abbreviated ──
    "phy exam tmrw 5pm add it",
    "maths hmwk nxt monday pls add",
    "sched cs lab friday 2pm",
    "yo add db project tuesday",
    "bro delete phy exam",
    "pls show tasks for tmrw",
    "hw due tmrw add it",

    # ── Complex multi-entity ──
    "add fyp meeting on wednesday at 2pm for 1hr with Ali high priority",
    "schedule cs lab on friday at 3pm duration 2 hours",
    "add phy exam monday 10am priority urgent with Hassan",
    "reschedule math exam from monday to friday at 4pm",
    "update phy exam priority to high",
    "rename cs lab to db project on tuesday",

    # ── Date/time variety ──
    "add bio exam on nxt monday",
    "cs lab day after tomorrow 3pm",
    "add quiz 1 in 3 days",
    "schedule presentation end of week",
    "add exam a week from now",

    # ── Show/query patterns ──
    "what do i have on monday",
    "show high priority tasks",
    "anything on friday",
    "list all pending tasks",
    "when is fyp meeting",

    # ── Update patterns ──
    "change math exam time to 5pm",
    "move fyp meeting to thursday",
    "mark all done tasks as complete",
    "extend cs lab to 2hr",
    "monday phy exam reschedule to wednesday",

    # ── Delete patterns ──
    "on tuesday delete phy exam",
    "tuesday phy exam delete",
    "phy exam delete",
    "remove all urgent tasks",
    "delete done tasks",
]

In [9]:
for text in inputs:
    entities = extract_entities(text, model, tokenizer, id2label)
    print(f'Input : "{text}"')
    if entities:
        for value, entity in entities:
            print(f'  {entity:<20}: "{value}"')
    else:
        print('  (no entities found)')
    print()

Input : "add math exam on monday high priority"
  task_name           : "math exam"
  date                : "monday"
  priority            : "high"

Input : "urgent add phy exam tomorrow"
  priority            : "urgent"
  task_name           : "phy exam"
  date                : "tomorrow"

Input : "low priority delete cs lab"
  priority            : "low"
  task_name           : "lab"

Input : "add fyp meeting friday top priority"
  day                 : "fyp"
  date                : "friday"
  priority            : "top"

Input : "add bio exam tuesday medium"
  task_name           : "bio exam"
  date                : "tuesday"
  priority            : "medium"

Input : "mark physics exam as done"
  task_name           : "physics exam"
  new_status          : "done"

Input : "mark cs lab complete"
  task_name           : "lab"
  new_status          : "complete"

Input : "physics exam done mark it"
  task_name           : "physics exam"
  status              : "done"

Input : "all pendi

In [ ]:
# ── Position swap test — same intent, different word orders ────────
swap_groups = [
    [
        "add urdu exam on monday",
        "monday urdu exam add it",
        "urdu exam monday",
    ],
    [
        "add cs lab with Ali on tuesday",
        "tuesday cs lab with Ali",
        "add lab report collaborator is Hassan",
    ],
]

for group in swap_groups:
    print('=' * 60)
    for text in group:
        entities = extract_entities(text, model, tokenizer, id2label)
        ent_str  = ', '.join(f'{e}="{v}"' for v, e in entities) if entities else '(none)'
        print(f'  "{text}"')
        print(f'  -> {ent_str}')
    print()